In [ ]:
import torch
import cv2
import numpy as np
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor


# 配置路徑
checkpoint = "../sam2/checkpoints/sam2.1_hiera_large.pt"
model_cfg  = "configs/sam2.1/sam2.1_hiera_l.yaml"
# -*- coding: utf-8 -*-

import os
import cv2
import scipy

from ultralytics import YOLO

# 載入分割模型
model = YOLO("yolov8m-seg.pt")

predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

def extract_mask_compare(image_path):
    image_name = os.path.basename(image_path)
    # 推論圖片
    results = model(image_path, conf=0.15)

    # 如果想儲存結果圖：
    box = None
    for result in results:
        for obj in result.summary():
            if obj["name"] == "bed":
                result.save(filename= "results/" + image_name.replace('.jpg', "_output1.jpg"))
                box = obj["box"]
    if box != None:
        # 載入圖像
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"無法載入圖像: {image_path}")

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # find the pixel point in the bounding box where the pixel has the most common color
        input_point = np.array([[(box["x1"] + box["x2"])//2 - (box["x1"] + box["x2"])//8, (box["y1"]+box["y2"])//2 - (box["y1"]+box["y2"])//8], 
                                [(box["x1"] + box["x2"])//2 - (box["x1"] + box["x2"])//8, (box["y1"]+box["y2"])//2 + (box["y1"]+box["y2"])//8],
                                [(box["x1"] + box["x2"])//2 + (box["x1"] + box["x2"])//8, (box["y1"]+box["y2"])//2 - (box["y1"]+box["y2"])//8],
                                [(box["x1"] + box["x2"])//2 + (box["x1"] + box["x2"])//8, (box["y1"]+box["y2"])//2 + (box["y1"]+box["y2"])//8]])
        input_label = np.array([1, 1, 1, 1])

        predictor.set_image(image_rgb)
        masks, scores, logits = predictor.predict(
            point_coords=input_point,
            point_labels=input_label,
            multimask_output=True # 會自動選擇分數最高的
        )
        best_mask_idx = np.argmax(scores)
        best_mask = masks[best_mask_idx]
        best_score = scores[best_mask_idx]
        print(image_rgb.dot(best_mask.T))

        # save the best mask to file
        mask_filename = image_name.replace('.jpg', "_mask.jpg")
        cv2.imwrite("results/" + mask_filename, (best_mask * 255).astype(np.uint8))

img_files = os.listdir("bed-images")
for img_file in img_files:
    if img_file.endswith('.jpg'):
        image_path = os.path.join("bed-images", img_file)
        extract_mask_compare(image_path)
        print(f"Processed {img_file}")


image 1/1 /home/itrib40351/Documents/GitHub/bedSheetFoldingRobot/bed-images/IMG_1221.jpg: 480x640 1 bed, 1 book, 16.8ms
Speed: 2.1ms preprocess, 16.8ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)
uint8 uint8
Processed IMG_1221.jpg

image 1/1 /home/itrib40351/Documents/GitHub/bedSheetFoldingRobot/bed-images/bed2.jpg: 480x640 1 bed, 16.4ms
Speed: 1.4ms preprocess, 16.4ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)
uint8 uint8
Processed bed2.jpg

image 1/1 /home/itrib40351/Documents/GitHub/bedSheetFoldingRobot/bed-images/bedsheets-french-fleur-bedsheet-blue-1.jpg: 640x640 2 potted plants, 1 bed, 5 vases, 21.6ms
Speed: 1.9ms preprocess, 21.6ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)
uint8 uint8
Processed bedsheets-french-fleur-bedsheet-blue-1.jpg

image 1/1 /home/itrib40351/Documents/GitHub/bedSheetFoldingRobot/bed-images/bed5.jpg: 640x640 1 chair, 1 potted plant, 1 bed, 1 vase, 20.8ms
Speed: 1.7ms preprocess, 20.8ms infer

In [2]:
# Calculate the mean RGB value of the cropped image
mean_rgb = np.mean(cropped_img.reshape(-1, 3), axis=0)

# Compute the Euclidean distance from each pixel to the mean color
distances = np.linalg.norm(cropped_img - mean_rgb, axis=-1)

# Find the pixel location with the minimum distance (closest to mean color)
mode_pixel_location = np.unravel_index(np.argmin(distances), distances.shape)
print(mode_pixel_location)

NameError: name 'cropped_img' is not defined